# 02 · Train tagger (encoder-only, gán nhãn từng ký tự)

Kaggle **GPU T4**. Input: `vi-diacritics-code` + output của notebook 01.
~5M tham số, 6 layer, d_model=256. Chạy ~5 tiếng, tự dừng và lưu checkpoint trước khi hết giờ.

In [ ]:
import glob, json, os, sys

CODE = glob.glob("/kaggle/input/**/vi_diacritics/__init__.py", recursive=True)[0].rsplit("/vi_diacritics", 1)[0]
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE
SPLITS = os.path.dirname(glob.glob("/kaggle/input/**/splits/train.txt", recursive=True)[0])
print(CODE, SPLITS)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q onnx onnxruntime

In [ ]:
!python -m vi_diacritics.train --task tagger --data {SPLITS} --out runs/tagger \
    --layers 6 --d-model 256 --heads 8 --d-ff 1024 --batch-size 128 --lr 5e-4 \
    --epochs 3 --eval-every 2000 --max-minutes 300

## Đường loss và accuracy trên val

In [ ]:
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in open("runs/tagger/log.jsonl", encoding="utf-8")]
tr = [r for r in rows if "loss" in r]
ev = [r["eval"] for r in rows if "eval" in r]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([r["step"] for r in tr], [r["loss"] for r in tr])
ax[0].set_yscale("log"); ax[0].set_title("train loss"); ax[0].set_xlabel("step")
ax[1].plot([e["step"] for e in ev], [e["word_acc"] for e in ev], label="word acc")
ax[1].plot([e["step"] for e in ev], [e["sent_acc"] for e in ev], label="sent acc")
ax[1].legend(); ax[1].set_title("val"); ax[1].set_xlabel("step")
fig.savefig("runs/tagger/curves.png", dpi=110, bbox_inches="tight")

## Xuất ONNX (fp32 + int8)

In [ ]:
!python -m vi_diacritics.export_onnx --run runs/tagger --out export
!ls -la export

## Attention: "con ma" vs "mà thôi" vs "ma túy"
Ký tự 'a' trong "ma" nhìn vào đâu để quyết định dấu?

In [ ]:
import torch
from vi_diacritics.model import TaggerTransformer
from vi_diacritics.viz import plot_attention
from vi_diacritics.vocab import CharVocab

cfg = json.load(open("runs/tagger/config.json"))
vocab = CharVocab.load("runs/tagger/vocab.json")
model = TaggerTransformer(len(vocab), cfg["n_labels"], torch.tensor(vocab.label_mask()), cfg["d_model"],
                          cfg["heads"], cfg["layers"], cfg["d_ff"], 0.0)
model.load_state_dict(torch.load("runs/tagger/best.pt", map_location="cpu")["model"])

for i, s in enumerate(["nhà có con ma", "ma tôi không biết", "ma túy là chất cấm"]):
    pos = s.index("ma") + 1
    plot_attention(model, vocab, s, layers=(0, 2, 5), query_char=pos, path=f"runs/tagger/attn_{i}.png")